# Pipeline

Main pipeline notebook. All logic lives in `pipeline/`; this notebook handles configuration and orchestration only.

## 1. Setup

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
%reload_ext autoreload

In [3]:
import pandas as pd
import os
import sys

from pathlib import Path

# Moving up to the project root to ensure imports work correctly regardless of execution context
project_root = Path(os.getcwd()).parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))


from pathlib import Path
from openai import OpenAI
from utilities import (
    OPENAI_API_KEY,
    TASK_STATEMENTS_PATH,
    TASK_RATINGS_PATH,
    MAJOR_CATEGORIES,
    WORK_RELATED_OUTPUT_PATH,
    TIMEZONES_OUTPUT_PATH,
    TASK_MAPPING_OUTPUT_PATH,
    LABOR_TRANSFER_OUTPUT_PATH,
    JOB_ZONES_PATH,
    FINAL_OUTPUT_PATH,
    ExecutionMode,
)
from pipeline import (
    load_wildchat,
    sample_conversations,
    preprocess_conversations,
    filter_work_conversations,
    find_timezones,
    normalize_timezone,
    map_conversation_to_task,
    filter_task_mappings,
    analyze_labor_transfer,
    expand_labor_transfer_labels,
)

In [4]:
# API client setup
client = OpenAI(api_key=OPENAI_API_KEY)

In [5]:
execution_mode = ExecutionMode.BATCH

## 2. Data Loading

In [6]:
english_conversations = load_wildchat()
total_rows = len(english_conversations)
print(f"Total English conversations: {total_rows}")

Loading dataset from disk:   0%|          | 0/46 [00:00<?, ?it/s]

Total English conversations: 1679371


In [7]:
sample_df = sample_conversations(english_conversations)
print(f"Sample shape: {sample_df.shape}")

Sample shape: (117555, 14)


In [66]:
processed_conversations_df = preprocess_conversations(sample_df)
print(f"After dedup: {processed_conversations_df.shape}")
processed_conversations_df.head(2)

After dedup: (104171, 5)


,conversation,timestamp,country,state,hashed_ip
0,"[{'role': 'user', 'content': 'can you name any...",2025-03-12 09:33:27,United Kingdom,Royal Kensington and Chelsea,62d519d4f9104d3ad08f25664bf790a39805b5d1e39c21...
1,"[{'role': 'user', 'content': 'sir followings a...",2024-10-01 15:01:08,Pakistan,Khyber Pakhtunkhwa,65ee10fbdef827a0b2eeb88fd635b256ce0b54ddf02c73...


In [67]:
sample_conversations_df = processed_conversations_df.copy()

## 3. Work-Related Conversation Filtering

In [68]:
if not WORK_RELATED_OUTPUT_PATH.exists():
    answers = filter_work_conversations(
        client=client,
        conversations=sample_conversations_df,
        path=WORK_RELATED_OUTPUT_PATH,
        execution_mode=execution_mode,
    )
    answers_df = pd.DataFrame(
        {
            "conversation": sample_conversations_df["conversation"],
            "is_work_related_model": answers,
        }
    )
    answers_df.to_csv(WORK_RELATED_OUTPUT_PATH, index=False)
else:
    answers_df = pd.read_csv(WORK_RELATED_OUTPUT_PATH)

sample_conversations_df["conversation_str"] = sample_conversations_df[
    "conversation"
].astype(str)
answers_df["conversation_str"] = answers_df["conversation"].astype(str)
sample_conversations_df = sample_conversations_df.merge(
    answers_df, on="conversation_str", how="left"
)

work_related_df = sample_conversations_df[
    sample_conversations_df["is_work_related_model"] == "Yes"
].copy()

print(f"Work-related conversations: {work_related_df.shape}")
work_related_df.head(2)

Work-related conversations: (41379, 8)


,conversation_x,timestamp,country,state,hashed_ip,conversation_str,conversation_y,is_work_related_model
1,"[{'role': 'user', 'content': 'sir followings a...",2024-10-01 15:01:08,Pakistan,Khyber Pakhtunkhwa,65ee10fbdef827a0b2eeb88fd635b256ce0b54ddf02c73...,"[{'role': 'user', 'content': 'sir followings a...","[{'role': 'user', 'content': 'sir followings a...",Yes
5,"[{'role': 'user', 'content': 'System: You are ...",2024-11-04 01:14:03,NaN,NaN,738c8bec3e3b3fe22bedc4885fb26c230754e22747e031...,"[{'role': 'user', 'content': ""System: You are ...","[{'role': 'user', 'content': ""System: You are ...",Yes


In [69]:
work_related_df.drop(columns=["conversation_y"], inplace=True)
work_related_df.rename(columns={"conversation_x": "conversation"}, inplace=True)
work_related_df.columns

Index(['conversation', 'timestamp', 'country', 'state', 'hashed_ip',
       'conversation_str', 'is_work_related_model'],
      dtype='str')

## 4. Timezone conversion

In [70]:
if not TIMEZONES_OUTPUT_PATH.exists():
    work_related_df = find_timezones(df=work_related_df)
    work_related_df.to_csv(TIMEZONES_OUTPUT_PATH, index=False)
else:
    work_related_df = pd.read_csv(TIMEZONES_OUTPUT_PATH)

work_related_df = work_related_df[work_related_df["timezone"].notnull()]
print(f"After timezone filter: {work_related_df.shape}")

After timezone filter: (40886, 8)


In [71]:
work_related_df = normalize_timezone(df=work_related_df)
work_related_df[["timestamp", "timezone", "timestamp_local"]].head(3)

,timestamp,timezone,timestamp_local
0,2024-10-01 15:01:08+00:00,Asia/Karachi,2024-10-01 20:01:08+05:00
1,2024-11-04 01:14:03+00:00,Asia/Kabul,2024-11-04 05:44:03+04:30
2,2024-11-05 23:54:06+00:00,Atlantic/Reykjavik,2024-11-05 23:54:06+00:00


In [72]:
work_related_df.shape

(40886, 9)

## 5. Task Mapping

In [73]:
tasks_ratings_df = pd.read_excel(TASK_RATINGS_PATH)
tasks_ratings_df.drop(
    columns=[
        "Domain Source",
        "Date",
        "Recommend Suppress",
        "Scale Name",
        "Category",
        "N",
        "Standard Error",
        "Lower CI Bound",
        "Upper CI Bound",
    ],
    inplace=True,
)
tasks_ratings_df = tasks_ratings_df[tasks_ratings_df["Scale ID"] == "IM"].copy()
tasks_ratings_df["importance_rank"] = tasks_ratings_df.groupby("Title")[
    "Data Value"
].rank(pct=True)
tasks_ratings_df.head(5)

,O*NET-SOC Code,Title,Task ID,Task,Scale ID,Data Value,importance_rank
7,11-1011.00,Chief Executives,8823,Direct or coordinate an organization's financi...,IM,4.54,1.000000
16,11-1011.00,Chief Executives,8831,Appoint department heads or managers and assig...,IM,4.48,0.967742
25,11-1011.00,Chief Executives,8825,Analyze operations to evaluate performance of ...,IM,4.40,0.935484
34,11-1011.00,Chief Executives,8826,"Direct, plan, or implement policies, objective...",IM,4.39,0.903226
43,11-1011.00,Chief Executives,8827,"Prepare budgets for approval, including those ...",IM,4.17,0.838710


In [74]:
tasks_df = pd.read_csv(TASK_STATEMENTS_PATH)
tasks_df.drop(
    columns=["Incumbents Responding", "Date", "Domain Source", "Task Type"],
    inplace=True,
)
tasks_df["major_category"] = tasks_df["O*NET-SOC Code"].apply(
    lambda x: MAJOR_CATEGORIES[x[0:2]]
)
tasks_df.head()

,O*NET-SOC Code,Title,Task ID,Task,major_category
0,11-1011.00,Chief Executives,8823,Direct or coordinate an organization's financi...,Management Occupations
1,11-1011.00,Chief Executives,8831,Appoint department heads or managers and assig...,Management Occupations
2,11-1011.00,Chief Executives,8825,Analyze operations to evaluate performance of ...,Management Occupations
3,11-1011.00,Chief Executives,8826,"Direct, plan, or implement policies, objective...",Management Occupations
4,11-1011.00,Chief Executives,8827,"Prepare budgets for approval, including those ...",Management Occupations


In [75]:
if not TASK_MAPPING_OUTPUT_PATH.exists():
    task_mapped_df = map_conversation_to_task(
        client=client,
        conversations=work_related_df,
        tasks=tasks_df,
        path=TASK_MAPPING_OUTPUT_PATH,
        execution_mode=execution_mode,
    )
else:
    task_mapped_df = pd.read_csv(TASK_MAPPING_OUTPUT_PATH)

print(f"Task mapped conversations: {task_mapped_df.shape}")
task_mapped_df.head(2)

Task mapped conversations: (40886, 3)


,conversation,professions,tasks
0,"[{'role': 'user', 'content': 'sir followings a...","['Editors', 'Technical Writers', 'Copy Writers...","['Editors: Prepare, rewrite and edit copy to i..."
1,"[{'role': 'user', 'content': ""System: You are ...","['Interpreters and Translators', 'Technical Wr...",['Interpreters and Translators: Read written m...


In [76]:
task_mapped_df = filter_task_mappings(df=task_mapped_df, column_name="tasks")

task_mapped_df["job_title"] = task_mapped_df["tasks"].apply(
    lambda x: x.split(":")[0] if pd.notnull(x) else None
)
task_mapped_df["selected_task"] = task_mapped_df["tasks"].apply(
    lambda x: x.split(":")[1] if pd.notnull(x) else None
)

print(f"After consensus filter: {task_mapped_df.shape}")
task_mapped_df.head(2)

After consensus filter: (28338, 5)


,conversation,professions,tasks,job_title,selected_task
1,"[{'role': 'user', 'content': ""System: You are ...","['Interpreters and Translators', 'Technical Wr...",Interpreters and Translators: Read written mat...,Interpreters and Translators,"Read written materials, such as legal documen..."
2,"[{'role': 'user', 'content': 'You are a helpfu...",['Market Research Analysts and Marketing Speci...,Market Research Analysts and Marketing Special...,Market Research Analysts and Marketing Special...,Conduct research on consumer opinions and mar...


In [77]:
task_mapped_df["conversation_str"] = task_mapped_df["conversation"].astype(str)
work_related_df["conversation_str"] = work_related_df["conversation"].astype(str)

task_mapped_df = task_mapped_df.merge(
    work_related_df.drop(columns=["conversation"]), on="conversation_str", how="inner"
)

task_mapped_df = task_mapped_df.drop(columns=["conversation_str"])

print(f"Final task mapped DataFrame: {task_mapped_df.shape}")
task_mapped_df.head(2)

Final task mapped DataFrame: (28338, 12)


,conversation,professions,tasks,job_title,selected_task,timestamp,country,state,hashed_ip,is_work_related_model,timezone,timestamp_local
0,"[{'role': 'user', 'content': ""System: You are ...","['Interpreters and Translators', 'Technical Wr...",Interpreters and Translators: Read written mat...,Interpreters and Translators,"Read written materials, such as legal documen...",2024-11-04 01:14:03+00:00,NaN,NaN,738c8bec3e3b3fe22bedc4885fb26c230754e22747e031...,Yes,Asia/Kabul,2024-11-04 05:44:03+04:30
1,"[{'role': 'user', 'content': 'You are a helpfu...",['Market Research Analysts and Marketing Speci...,Market Research Analysts and Marketing Special...,Market Research Analysts and Marketing Special...,Conduct research on consumer opinions and mar...,2024-11-05 23:54:06+00:00,Denmark,Capital Region,1b8b85252359a2c246c3086a72367c7448601985693934...,Yes,Atlantic/Reykjavik,2024-11-05 23:54:06+00:00


## 6. Labor Transfer Analysis

In [78]:
if not LABOR_TRANSFER_OUTPUT_PATH.exists():
    labor_transfer_labels = analyze_labor_transfer(
        client=client, df=task_mapped_df, execution_mode=execution_mode
    )
    pd.DataFrame({"label": labor_transfer_labels}).to_csv(
        LABOR_TRANSFER_OUTPUT_PATH, index=False
    )
else:
    labor_transfer_labels = pd.read_csv(LABOR_TRANSFER_OUTPUT_PATH)["label"].tolist()

print(f"Labor transfer labels: {len(labor_transfer_labels)}")
task_mapped_df["labor_transfer"] = labor_transfer_labels
print(f"Labor transfer labels assigned: {task_mapped_df.shape}")

Labor transfer labels: 28338
Labor transfer labels assigned: (28338, 13)


In [79]:
task_mapped_df = expand_labor_transfer_labels(
    df=task_mapped_df, label_column="labor_transfer"
)
print(f"Final DataFrame: {task_mapped_df.shape}")
task_mapped_df.head(2)

Final DataFrame: (28338, 20)


,conversation,professions,tasks,job_title,selected_task,timestamp,country,state,hashed_ip,is_work_related_model,timezone,timestamp_local,interaction_type,task_match,label,lt1_reason,transferred_from,transferred_from_other,rationale,confidence
0,"[{'role': 'user', 'content': ""System: You are ...","['Interpreters and Translators', 'Technical Wr...",Interpreters and Translators: Read written mat...,Interpreters and Translators,"Read written materials, such as legal documen...",2024-11-04 01:14:03+00:00,NaN,NaN,738c8bec3e3b3fe22bedc4885fb26c230754e22747e031...,Yes,Asia/Kabul,2024-11-04 05:44:03+04:30,consumer,good,LT2,NaN,translator,NaN,The user asked for a direct English→Chinese tr...,high
1,"[{'role': 'user', 'content': 'You are a helpfu...",['Market Research Analysts and Marketing Speci...,Market Research Analysts and Marketing Special...,Market Research Analysts and Marketing Special...,Conduct research on consumer opinions and mar...,2024-11-05 23:54:06+00:00,Denmark,Capital Region,1b8b85252359a2c246c3086a72367c7448601985693934...,Yes,Atlantic/Reykjavik,2024-11-05 23:54:06+00:00,consumer,good,LT2,NaN,other,market_research_analyst / marketing_specialist,The user requests a structured market-research...,high


# 7. Job Zones

In [80]:
job_zones_df = pd.read_excel(JOB_ZONES_PATH)
job_zones_df.head(2)

,O*NET-SOC Code,Title,Job Zone,Date,Domain Source
0,11-1011.00,Chief Executives,5,08/2023,Analyst
1,11-1011.03,Chief Sustainability Officers,5,08/2021,Analyst


In [81]:
task_mapped_df = task_mapped_df.merge(
    job_zones_df[["Title", "Job Zone", "O*NET-SOC Code"]],
    left_on="job_title",
    right_on="Title",
    how="left",
)
task_mapped_df = task_mapped_df.drop(columns=["Title"])
print(f"After merging job zones: {task_mapped_df.shape}")
task_mapped_df.head(2)

After merging job zones: (28338, 22)


,conversation,professions,tasks,job_title,selected_task,timestamp,country,state,hashed_ip,is_work_related_model,...,interaction_type,task_match,label,lt1_reason,transferred_from,transferred_from_other,rationale,confidence,Job Zone,O*NET-SOC Code
0,"[{'role': 'user', 'content': ""System: You are ...","['Interpreters and Translators', 'Technical Wr...",Interpreters and Translators: Read written mat...,Interpreters and Translators,"Read written materials, such as legal documen...",2024-11-04 01:14:03+00:00,NaN,NaN,738c8bec3e3b3fe22bedc4885fb26c230754e22747e031...,Yes,...,consumer,good,LT2,NaN,translator,NaN,The user asked for a direct English→Chinese tr...,high,4.0,27-3091.00
1,"[{'role': 'user', 'content': 'You are a helpfu...",['Market Research Analysts and Marketing Speci...,Market Research Analysts and Marketing Special...,Market Research Analysts and Marketing Special...,Conduct research on consumer opinions and mar...,2024-11-05 23:54:06+00:00,Denmark,Capital Region,1b8b85252359a2c246c3086a72367c7448601985693934...,Yes,...,consumer,good,LT2,NaN,other,market_research_analyst / marketing_specialist,The user requests a structured market-research...,high,4.0,13-1161.00


# 8. Task Importance rating

In [82]:
tasks_ratings_df.columns

Index(['O*NET-SOC Code', 'Title', 'Task ID', 'Task', 'Scale ID', 'Data Value',
       'importance_rank'],
      dtype='str')

In [83]:
task_mapped_df.columns

Index(['conversation', 'professions', 'tasks', 'job_title', 'selected_task',
       'timestamp', 'country', 'state', 'hashed_ip', 'is_work_related_model',
       'timezone', 'timestamp_local', 'interaction_type', 'task_match',
       'label', 'lt1_reason', 'transferred_from', 'transferred_from_other',
       'rationale', 'confidence', 'Job Zone', 'O*NET-SOC Code'],
      dtype='str')

In [84]:
task_mapped_df["selected_task"] = task_mapped_df["selected_task"].str.strip()
tasks_ratings_df["Task"] = tasks_ratings_df["Task"].str.strip()

task_mapped_df = task_mapped_df.merge(
    tasks_ratings_df[["O*NET-SOC Code", "Task", "Title", "importance_rank"]],
    left_on=["job_title", "selected_task", "O*NET-SOC Code"],
    right_on=["Title", "Task", "O*NET-SOC Code"],
    how="left",
)
task_mapped_df.drop(columns=["Title", "Task"], inplace=True)

In [85]:
task_mapped_df.columns

Index(['conversation', 'professions', 'tasks', 'job_title', 'selected_task',
       'timestamp', 'country', 'state', 'hashed_ip', 'is_work_related_model',
       'timezone', 'timestamp_local', 'interaction_type', 'task_match',
       'label', 'lt1_reason', 'transferred_from', 'transferred_from_other',
       'rationale', 'confidence', 'Job Zone', 'O*NET-SOC Code',
       'importance_rank'],
      dtype='str')

In [87]:
final_df = task_mapped_df.copy()
final_df.to_csv(FINAL_OUTPUT_PATH, index=False)
final_df.shape

(28338, 23)